In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
from scipy.optimize import minimize
import warnings
warnings.filterwarnings("ignore")

In [ ]:
def obtener_datos(tickers, start_date):
    print(f"Descargando datos desde {start_date}...")
    data = yf.download(tickers, start=start_date)['Adj Close']
    return data.dropna()

In [ ]:
def optimizar_hibrido(data, div_yields_array, risk_free_rate=0.11, peso_rentas=2.5):
    log_returns = np.log(data / data.shift(1)).dropna()
    cov_matrix = log_returns.cov() * 252
    expected_returns = log_returns.mean() * 252
    num_assets = len(data.columns)

    # Función a minimizar: Sharpe Ratio modificado para premiar los dividendos
    def objetivo_hibrido(weights, expected_returns, cov_matrix, risk_free_rate, div_yields_array):
        p_ret = np.sum(expected_returns * weights)
        p_vol = np.sqrt(np.dot(weights.T, np.dot(cov_matrix, weights)))
        sharpe = (p_ret - risk_free_rate) / p_vol
        p_yield = np.sum(div_yields_array * weights)
        # Se penaliza el riesgo, se premia el retorno y se fuerza la búsqueda de flujo (peso_rentas)
        return -(sharpe + (peso_rentas * p_yield))

    constraints = ({'type': 'eq', 'fun': lambda x: np.sum(x) - 1})
    bounds = tuple((0, 1) for asset in range(num_assets))
    initial_weights = num_assets * [1. / num_assets,]

    optimized_result = minimize(objetivo_hibrido, initial_weights,
                                args=(expected_returns, cov_matrix, risk_free_rate, div_yields_array),
                                method='SLSQP', bounds=bounds, constraints=constraints)

    return optimized_result.x

In [ ]:
def calcular_y_proyectar(pesos, tickers, capital_mxn, precios, titulos_actuales, yields, freqs):
    print("\n" + "="*60)
    print(f"ASIGNACIÓN ÓPTIMA Y BALANCEADA PARA INVERTIR: ${capital_mxn:.2f} MXN")
    print("="*60)

    flujo_anual_total = 0

    for i, ticker in enumerate(tickers):
        asignacion_mxn = capital_mxn * pesos[i]
        titulos_comprar = int(asignacion_mxn // precios[ticker])
        titulos_totales = titulos_actuales[ticker] + titulos_comprar
        capital_total_invertido = titulos_totales * precios[ticker]

        # Cálculo de flujos
        yield_anual = yields[ticker]
        flujo_anual = capital_total_invertido * yield_anual
        flujo_anual_total += flujo_anual

        print(f"[{ticker}] - {freqs[ticker]}")
        print(f"  - Peso Algorítmico: {pesos[i]*100:.2f}% (Compras hoy: {titulos_comprar} títulos)")
        print(f"  - Inventario Total: {titulos_totales} títulos (Actuales: {titulos_actuales[ticker]} + Nuevos: {titulos_comprar})")

        if yield_anual > 0:
            print(f"  - Flujo Efectivo Esperado: ${flujo_anual:.2f} MXN Anuales", end="")
            if freqs[ticker] == 'Mensual':
                print(f" (~${flujo_anual/12:.2f} MXN al mes)")
            elif freqs[ticker] == 'Trimestral':
                print(f" (~${flujo_anual/4:.2f} MXN cada trimestre)")
            else:
                print(" (Acumulado/Irregular)")
        print("-" * 60)

    print(f"\n>>> FLUJO PASIVO TOTAL ESTIMADO DEL PORTAFOLIO: ${flujo_anual_total:.2f} MXN ANUALES <<<")
    print(f">>> EQUIVALENTE MENSUAL PROMEDIO: ${flujo_anual_total/12:.2f} MXN AL MES <<<\n")

if __name__ == '__main__':
    tickers = ['IVVPESO.MX', 'NAFTRAC.MX', 'FUNO11.MX', 'FMTY14.MX']

    # 1. EDITA AQUÍ TU INVENTARIO ACTUAL
    titulos_actuales = {
        'IVVPESO.MX': 3,
        'NAFTRAC.MX': 8,
        'FUNO11.MX': 1,
        'FMTY14.MX': 30
    }

    # Rendimientos por dividendos anualizados estimados (Yield)
    yields = {
        'IVVPESO.MX': 0.015, # ~1.5%
        'NAFTRAC.MX': 0.025, # ~2.5%
        'FUNO11.MX': 0.085,  # ~8.5%
        'FMTY14.MX': 0.082   # ~8.2%
    }

    frecuencias = {
        'IVVPESO.MX': 'Trimestral',
        'NAFTRAC.MX': 'Anual/Irregular',
        'FUNO11.MX': 'Trimestral',
        'FMTY14.MX': 'Mensual'
    }

    # Descarga de datos robusta para evitar el KeyError 'Adj Close'
    print(f"Descargando datos de yfinance...")
    df_download = yf.download(tickers, start='2023-01-01')

    if 'Adj Close' in df_download.columns:
        datos = df_download['Adj Close']
    elif 'adj close' in df_download.columns:
        datos = df_download['adj close']
    else:
        datos = df_download['Close'] if 'Close' in df_download.columns else df_download['close']

    datos = datos.dropna()
    precios_hoy = datos.iloc[-1]
    div_yields_array = np.array([yields[t] for t in tickers])

    capital = 28.25 # Capital a invertir hoy

    # Generar pesos forzando el balance rentas/crecimiento
    pesos_optimos = optimizar_hibrido(datos, div_yields_array)

    calcular_y_proyectar(pesos_optimos, tickers, capital, precios_hoy, titulos_actuales, yields, frecuencias)

[                       0%                       ]

Descargando datos de yfinance...


[*********************100%***********************]  4 of 4 completed


ASIGNACIÓN ÓPTIMA Y BALANCEADA PARA INVERTIR: $28.25 MXN
[IVVPESO.MX] - Trimestral
  - Peso Algorítmico: 0.00% (Compras hoy: 0 títulos)
  - Inventario Total: 3 títulos (Actuales: 3 + Nuevos: 0)
  - Flujo Efectivo Esperado: $7.02 MXN Anuales (~$1.75 MXN cada trimestre)
------------------------------------------------------------
[NAFTRAC.MX] - Anual/Irregular
  - Peso Algorítmico: 3.35% (Compras hoy: 0 títulos)
  - Inventario Total: 8 títulos (Actuales: 8 + Nuevos: 0)
  - Flujo Efectivo Esperado: $13.83 MXN Anuales (Acumulado/Irregular)
------------------------------------------------------------
[FUNO11.MX] - Trimestral
  - Peso Algorítmico: 96.65% (Compras hoy: 0 títulos)
  - Inventario Total: 1 títulos (Actuales: 1 + Nuevos: 0)
  - Flujo Efectivo Esperado: $2.82 MXN Anuales (~$0.71 MXN cada trimestre)
------------------------------------------------------------
[FMTY14.MX] - Mensual
  - Peso Algorítmico: 0.00% (Compras hoy: 0 títulos)
  - Inventario Total: 30 títulos (Actuales: 30 +